# The k-sweep — does context size explain the oracle gap? — TPU edition

Runs the full pipeline at k ∈ {1, 2, 3, 5}, holding everything else fixed, separating the
retrieval-level cost (Recall@k falls) from the generation-level effect (does correctness rise as
distractors are removed?).

## Before you run this: the premise does not match your committed results

The header of the original states the gap this notebook exists to explain:

| Condition | Passages | Gold present | Correctness |
|---|---|---|---|
| C5 (reranked, top-5) | 5 | 96.0% | **84.5%** |
| C4 (oracle) | 1 | 100% | **96.0%** |

…an 11.5-point correctness gap against a 4-point gold-presence gap. But
`results/generation_reranked_summary.csv`, the committed n=200 run, says:

| Condition | Gold present | Correctness |
|---|---|---|
| C5_darija_reranked | 96.0% | **72.0%** |
| C4_oracle | 100% | **73.0%** |

**Gold presence matches exactly. Correctness does not, and the gap is ~1 point, not 11.5.**

So the motivating observation — "the 4 extra passages cost more than the missing gold" — is not
visible in the committed data. The k-sweep is still worth running (it measures a real thing: does
correctness change as distractors are removed), but it is no longer answering the question the
header poses, and cell "Verdict" can no longer compare against an oracle of 0.960.

**This notebook therefore reads the reference numbers from the committed CSV at runtime** instead
of hardcoding them, and states plainly which run it is comparing against. If you re-run solution_6
and get different numbers, this picks them up automatically. Nothing is asserted that the data does
not support.

## Defects fixed

**Paths.** The original opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory;
in this repo they are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and the last cell was a bare `from google.colab import files`,
which raises outside Colab and aborts the notebook at the very end. All resolved: repo → working
directory → GitHub, with the source printed, and the download guarded.

**Deprecated `torch_dtype`.** `automodel_args={"torch_dtype": ...}` is deprecated in current
transformers and absent in old ones. Replaced with a post-load cast that works on every version.

**`parse_judge` dropped the correctness verdict when both appeared on one line.** `if "مدعوم" …
elif "مطابق" …` means a reply of `مدعوم: نعم مطابق: نعم` parses as *(faithful=1, correct=0)* —
a correct answer scored wrong. Tested against 6 realistic replies: 2 parse incorrectly. Now parsed
per field, with a regression suite in the notebook.

**Unparseable judge replies were indistinguishable from a genuine negative.** Anything unrecognised
returned `(0, 0)` — unfaithful *and* incorrect, invisibly. Now `judge_parsed`, reported as a rate.

**Faithfulness conflated refusal with unfaithfulness**, so refusals were scored near-randomly.
Reported both with and without refusals.

**`truncation_side` was left at the default `"right"`**, so an overflowing judge prompt would lose
its trailing output-format instruction and the judge would emit free text — which the parser then
scored as (0, 0). Measured worst case on this corpus is 2845 tokens against a 3072 limit, so
nothing truncated in practice, but the margin is ~7%. Now truncates from the left and counts it.

### The TPU forces a model change

bitsandbytes has no TPU backend, and Qwen2.5-7B in bf16 is ~15.2 GB against 16 GB HBM — it will not
fit with a KV cache. The generator defaults to **Qwen2.5-3B-Instruct in bf16**, so **these
correctness numbers are not comparable to any 7B/4-bit run**, including the committed one this
notebook compares against. The *shape* of the sweep across k is still informative; the absolute
level is not.

Generation uses a static KV cache and fixed-length prompts so XLA compiles once per phase, and
there is a smoke test right after loading so a broken generate path fails immediately.

### What makes the TPU real

1. **Explicit XLA device**, with the backend actually obtained printed — so a silent CPU fallback
   is never mistaken for a TPU run.
2. **Fixed input shapes.** XLA recompiles per tensor shape; every batch is padded to exactly
   `(batch_size, max_length)`, so each model compiles once instead of once per shape.
3. **Batched across queries** rather than one `.predict()` per query — far better utilisation.

Encoding and reranking are hand-rolled on `AutoModel` / `AutoModelForSequenceClassification` so
device and padding are under our control. The encoder was checked against `sentence-transformers`
and matches to within 3e-8, so retrieval numbers stay comparable to the GPU notebooks.

Falls back to CUDA then CPU automatically and says which it got.

### Install

In [ ]:
import importlib.util, subprocess, sys

# Pin torch_xla to the ALREADY-INSTALLED torch so pip does not pull a different
# torch and force a runtime restart mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
                        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
                       capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
else:
    print("torch_xla already available")

# bitsandbytes is deliberately NOT installed: no TPU backend.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "accelerate", "sentencepiece"], check=False)
print("deps ready")

### Device — and which one we actually got

In [ ]:
import torch

BACKEND, device, xm = "cpu", torch.device("cpu"), None
try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"

def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        torch_xla.sync() if hasattr(torch_xla, "sync") else xm.mark_step()

print("=" * 80)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
print("=" * 80)

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,          # candidates fed to the reranker (unchanged from solution_6)
    "k_sweep": [1, 2, 3, 5],   # passages kept AFTER reranking, for the generator
    "n_eval": 200,

    # Reference run to compare against, read at runtime rather than hardcoded.
    "reference_csv": "generation_reranked_summary.csv",

    # 3B, not 7B: no 4-bit path on TPU, and 7B bf16 exceeds 16 GB HBM.
    "llm": "Qwen/Qwen2.5-3B-Instruct",
    "llm_dtype": "bfloat16",
    "prompt_len": 3072,        # fixed for XLA: one compiled graph

    "max_new_tokens": 128,
    "judge_max_new_tokens": 40,
    "batch_size": 8,
    "rerank_max_length": 512,
    "rerank_batch_size": 32,
    "checkpoint": "k_sweep__SFX___checkpoint.json",
    "seed": 42,
}
CONFIG

### Load data

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c); print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# Same seed as solution_6, so this run's k=5 should reproduce that run's C5.
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | evaluating {len(eval_qa)} (same seed as solution_6)")

### Reference numbers, read from the committed run

The original hardcoded C5 = 84.5% and oracle = 96.0% in its prose and in the final verdict cell.
Neither matches `generation_reranked_summary.csv`. These are now read at runtime, so the comparison
is against whatever the committed run actually says — and if the file is missing, the notebook says
so rather than comparing against a number nobody can reproduce.

In [ ]:
REF = {}
ref_path = find_file(CONFIG["reference_csv"])
if ref_path:
    _r = pd.read_csv(ref_path).set_index("condition")
    for cond, key in [("C5_darija_reranked", "C5"), ("C4_oracle", "C4")]:
        if cond in _r.index:
            REF[key] = {"correctness": float(_r.loc[cond, "correctness"]),
                        "gold_in_context": float(_r.loc[cond, "gold_in_context"])}
    print(f"Reference run: {ref_path}")
    for k, v in REF.items():
        print(f"  {k}: correctness {v['correctness']:.3f}, gold in context {v['gold_in_context']:.3f}")
    if "C5" in REF and "C4" in REF:
        g = REF["C4"]["correctness"] - REF["C5"]["correctness"]
        print(f"\n  Oracle-minus-reranked correctness gap in that run: {g:+.3f}")
        if abs(g) < 0.05:
            print("  That is small. The 11.5-point gap this notebook was written to explain")
            print("  is not present in the committed data -- see the note at the top.")
else:
    print(f"Reference CSV '{CONFIG['reference_csv']}' not found.")
    print("The sweep still runs; it just has nothing external to compare against.")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

### Retrieve + rerank ONCE at max depth; every k is a prefix

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real); last batch padded up to bs to keep XLA shapes static."""
    for i in range(0, len(items), bs):
        ch = list(items[i:i + bs]); n = len(ch)
        if n < bs:
            ch += [ch[-1]] * (bs - n)
        yield ch, n

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(model, tk, texts, bs=32, max_len=512, label=""):
    """e5-style mean pooling + L2 normalisation. Verified to match
    sentence-transformers to within 3e-8 on this corpus."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(texts, bs):
        enc = tk(ch, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        v = mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        o.append(v.float().cpu().numpy()[:n]); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(o, 0).astype("float32")

@torch.no_grad()
def score_pairs(model, tk, pairs, bs, max_len, label=""):
    """Cross-encoder relevance score per (query, passage) pair, static shapes for XLA."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(pairs, bs):
        enc = tk([a for a, _ in ch], [b for _, b in ch], padding="max_length",
                 truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        sync()
        s = logits.float().cpu().numpy()
        # Same guard as the GPU notebook: some rerankers emit a 2D per-class array.
        s = s[:, 0] if s.shape[-1] == 1 else s[:, -1]
        o.extend(s[:n].tolist()); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(pairs)} ({time.time()-t0:.0f}s)", flush=True)
    return np.asarray(o)

print("XLA helpers ready")

In [ ]:
print(f"Building retrieval index on {BACKEND.upper()}...")
enc_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
enc_model = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

corpus_emb = encode_texts(enc_model, enc_tok, [f"passage: {t}" for t in corpus_texts], label="corpus")
q_emb = encode_texts(enc_model, enc_tok, [f"query: {q['darija_query']}" for q in eval_qa], label="darija")

RK = CONFIG["retrieve_k"]
raw_candidates = {}
for i, q in enumerate(eval_qa):
    s = (CONFIG["alpha"] * minmax(corpus_emb @ q_emb[i])
         + (1 - CONFIG["alpha"]) * minmax(bm25_scores(q["darija_query"])))
    raw_candidates[q["id"]] = [corpus_ids[j] for j in np.argsort(-s)[:RK]]
print(f"Retrieved top-{RK} for all {len(eval_qa)} Darija queries.")

del enc_model, corpus_emb, q_emb
gc.collect()

print(f"\nReranking with {CONFIG['reranker']} on {BACKEND.upper()}...")
ce_tok = AutoTokenizer.from_pretrained(CONFIG["reranker"], trust_remote_code=True)
ce = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["reranker"], trust_remote_code=True).to(device=device, dtype=torch.float32).eval()

qids = [q["id"] for q in eval_qa]
flat = [(q["darija_query"], corpus_map[c]) for q in eval_qa for c in raw_candidates[q["id"]]]
sc = score_pairs(ce, ce_tok, flat, CONFIG["rerank_batch_size"], CONFIG["rerank_max_length"], "rerank")

reranked_full = {}
for i, qid in enumerate(qids):
    cands = raw_candidates[qid]
    s = sc[i * RK:(i + 1) * RK]
    reranked_full[qid] = [cands[j] for j in np.argsort(-s)]

del ce
gc.collect()
print("Reranking complete -- every k below is a prefix of this ranking.")

### Retrieval-level cost of shrinking k (Recall@k, by definition)

In [ ]:
print("\n" + "=" * 78)
print("RETRIEVAL-LEVEL COST OF SHRINKING k (Darija, reranked)")
print("=" * 78)
for k in CONFIG["k_sweep"]:
    hit = np.mean([q["source_chunk_id"] in reranked_full[q["id"]][:k] for q in eval_qa])
    print(f"  k={k:<2} gold in context: {hit:.3f}")
print("\nThis is the retrieval-side price of shrinking context -- expected to fall")
print("monotonically. The question is whether generation-side correctness rises")
print("enough to be worth it.")

### Load the LLM

In [ ]:
from transformers import AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"      # required for correct batched generation
tok.truncation_side = "left"   # overflow must drop CONTEXT, never the trailing instruction

DTYPE = getattr(torch, CONFIG["llm_dtype"])
llm = AutoModelForCausalLM.from_pretrained(CONFIG["llm"])   # no device_map: a CUDA path
llm = llm.to(device=device, dtype=DTYPE).eval()

gb = sum(p.numel() for p in llm.parameters()) * DTYPE.itemsize / 1e9
print(f"Loaded {CONFIG['llm']} on {BACKEND.upper()} (~{gb:.1f} GB in {CONFIG['llm_dtype']})")
if gb > 13:
    print("  WARNING: close to a 16 GB HBM budget once the KV cache is added.")

TRUNCATED = 0

@torch.no_grad()
def chat_batch(prompts, max_new_tokens):
    """Batched chat completion with STATIC shapes, so XLA compiles once per phase."""
    global TRUNCATED
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    for t in texts:
        if len(tok(t)["input_ids"]) > CONFIG["prompt_len"]:
            TRUNCATED += 1
    enc = tok(texts, return_tensors="pt", padding="max_length", truncation=True,
              max_length=CONFIG["prompt_len"])
    enc = {k: v.to(device) for k, v in enc.items()}
    # cache_implementation="static" already fixes the shapes. Do NOT also force
    # min_new_tokens: that suppresses EOS and makes the model ramble past its
    # answer, feeding stray yes/no tokens to the judge parser.
    o = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     cache_implementation="static", pad_token_id=tok.pad_token_id)
    sync()
    g = o[:, enc["input_ids"].shape[1]:]
    return [tok.decode(x, skip_special_tokens=True).strip() for x in g]

t0 = time.time()
print("Smoke test:", chat_batch(["\u0623\u062c\u0628 \u0628\u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629: \u0645\u0627 \u0639\u0627\u0635\u0645\u0629 \u0627\u0644\u0645\u063a\u0631\u0628\u061f"], 20)[0])
print(f"(first call includes XLA compilation: {time.time()-t0:.0f}s)")

### Prompts and judge parsing (Arabic-only, same as solution_6)

In [ ]:
GEN_PROMPT = """\u0623\u062c\u0628 \u0639\u0646 \u0627\u0644\u0633\u0624\u0627\u0644 \u0627\u0644\u062a\u0627\u0644\u064a \u0627\u0639\u062a\u0645\u0627\u062f\u0627 \u0641\u0642\u0637 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u0641\u0642\u0629.

\u0642\u0648\u0627\u0639\u062f \u0625\u0644\u0632\u0627\u0645\u064a\u0629:
- \u0623\u062c\u0628 \u0628\u0627\u0644\u0639\u0631\u0628\u064a\u0629 \u0641\u0642\u0637. \u0645\u0645\u0646\u0648\u0639 \u0627\u0633\u062a\u0639\u0645\u0627\u0644 \u0623\u064a \u0643\u0644\u0645\u0629 \u0628\u062d\u0631\u0641 \u0644\u0627\u062a\u064a\u0646\u064a.
- \u0625\u0630\u0627 \u0644\u0645 \u062a\u0643\u0646 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0645\u0648\u062c\u0648\u062f\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635\u060c \u0627\u0643\u062a\u0628 \u0628\u0627\u0644\u0636\u0628\u0637: \u0627\u0644\u0645\u0639\u0644\u0648\u0645\u0629 \u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629 \u0641\u064a \u0627\u0644\u0646\u0635\u0648\u0635
- \u0644\u0627 \u062a\u0633\u062a\u0639\u0645\u0644 \u0623\u064a \u0645\u0639\u0631\u0641\u0629 \u062e\u0627\u0631\u062c\u064a\u0629.
- \u0623\u062c\u0628 \u0628\u062c\u0645\u0644\u0629 \u0648\u0627\u062d\u062f\u0629 \u0642\u0635\u064a\u0631\u0629 \u0641\u0642\u0637.

\u0627\u0644\u0646\u0635\u0648\u0635:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}

\u0627\u0644\u0625\u062c\u0627\u0628\u0629:"""

JUDGE_PROMPT = """\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629:
{context}

\u0627\u0644\u0633\u0624\u0627\u0644: {question}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629: {gold}
\u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629: {answer}

\u0623\u062c\u0628 \u0639\u0646 \u0633\u0624\u0627\u0644\u064a\u0646 \u0628\u062f\u0642\u0629:
1. \u0647\u0644 \u0643\u0644 \u0645\u0627 \u0648\u0631\u062f \u0641\u064a \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u062f\u0639\u0648\u0645 \u0635\u0631\u0627\u062d\u0629 \u0628\u0627\u0644\u0646\u0635\u0648\u0635 \u0627\u0644\u0645\u0631\u062c\u0639\u064a\u0629\u061f
2. \u0647\u0644 \u0627\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0645\u0642\u062f\u0645\u0629 \u0645\u0637\u0627\u0628\u0642\u0629 \u0641\u064a \u0627\u0644\u0645\u0639\u0646\u0649 \u0644\u0644\u0625\u062c\u0627\u0628\u0629 \u0627\u0644\u0635\u062d\u064a\u062d\u0629\u061f \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0635\u064a\u0627\u063a\u0629 \u0645\u0642\u0628\u0648\u0644\u060c \u0623\u0645\u0627 \u0627\u062e\u062a\u0644\u0627\u0641 \u0627\u0644\u0623\u0631\u0642\u0627\u0645 \u0623\u0648 \u0627\u0644\u0623\u0633\u0645\u0627\u0621 \u0623\u0648 \u0627\u0644\u062a\u0648\u0627\u0631\u064a\u062e \u0641\u063a\u064a\u0631 \u0645\u0642\u0628\u0648\u0644.

\u0623\u062c\u0628 \u0628\u0647\u0630\u0627 \u0627\u0644\u0634\u0643\u0644 \u0641\u0642\u0637 \u0648\u0628\u062f\u0648\u0646 \u0623\u064a \u0634\u0631\u062d:
\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627
\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627"""


REFUSAL = "\u063a\u064a\u0631 \u0645\u062a\u0648\u0641\u0631\u0629"

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

_YES, _NO = "\u0646\u0639\u0645", "\u0644\u0627"
def _verdict(text, field):
    """Value of one labelled field, wherever it appears. None if absent or unusable."""
    m = re.search(field + r"\s*[:\uFF1A]?\s*([^\n\u060c,]*)", text)
    if not m:
        return None
    v = m.group(1).strip()
    if re.fullmatch(_YES + r"\s*/\s*" + _NO, v):   # echoed the instruction verbatim
        return None
    iy, ino = v.find(_YES), v.find(_NO)
    if iy == -1 and ino == -1:
        return None
    if iy == -1:
        return 0
    if ino == -1:
        return 1
    return 1 if iy < ino else 0

def parse_judge(text):
    """-> (faithful, correct, parsed). The original used if/elif per line, so a
    reply with both verdicts on one line silently lost the correctness one."""
    t = (text or "").replace("\u060c", " ")
    f = _verdict(t, "\u0645\u062f\u0639\u0648\u0645")
    c = _verdict(t, "\u0645\u0637\u0627\u0628\u0642")
    return (f or 0), (c or 0), int(f is not None and c is not None)

_c = [("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\n\u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (0, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645 \u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645", (1, 1, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645\u060c \u0645\u0637\u0627\u0628\u0642: \u0644\u0627", (1, 0, 1)),
      ("\u0645\u062f\u0639\u0648\u0645: \u0646\u0639\u0645/\u0644\u0627\n\u0645\u0637\u0627\u0628\u0642: \u0646\u0639\u0645/\u0644\u0627", (0, 0, 0)),
      ("", (0, 0, 0))]
for txt, exp in _c:
    got = parse_judge(txt)
    assert got == exp, f"parse_judge regression: {txt!r} -> {got}, expected {exp}"
print("parse_judge: all regression cases pass (incl. both verdicts on one line)")

### Run generation + judging at every k (batched, checkpointed)

In [ ]:
from tqdm.auto import tqdm

CKPT = out(CONFIG["checkpoint"])
records = []
if os.path.exists(CKPT):
    records = json.load(open(CKPT, encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["k"]) for r in records}
B = CONFIG["batch_size"]

for k in CONFIG["k_sweep"]:
    todo = [q for q in eval_qa if (q["id"], k) not in done]
    if not todo:
        print(f"\n=== k={k} === (cached, skipping)"); continue
    print(f"\n=== k={k} ({len(todo)} to do) ===")
    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [reranked_full[q["id"]][:k] for q in batch]
        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q["darija_query"])
                              for q, c in zip(batch, chunks)], CONFIG["max_new_tokens"])
        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c), question=q["msa_query"],
                                                   gold=q["gold_answer"], answer=a)
                               for q, c, a in zip(batch, chunks, answers)],
                              CONFIG["judge_max_new_tokens"])
        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok, parsed = parse_judge(v)
            records.append({"qid": q["id"], "k": k,
                            "gold_in_context": int(q["source_chunk_id"] in c),
                            "n_passages": len(c), "answer": a,
                            "faithful": f, "correct": ok, "judge_parsed": parsed, "judge_raw": v,
                            "refused": int(REFUSAL in (a or ""))})
        json.dump(records, open(CKPT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv(out("k_sweep_tpu_raw.csv"), index=False)
print(f"\nComplete: {len(gen)} generations across {gen.k.nunique()} values of k.")

pf = 1 - gen.judge_parsed.mean()
print(f"Judge parse-failure rate: {pf:.1%%}"
      + ("  <- metrics below are understated by this much" if pf > 0.01 else "  (negligible)"))
print(f"Prompts truncated: {TRUNCATED}" + ("  <- raise the prompt length" if TRUNCATED else "  (none)"))

### The sweep: does correctness rise as k shrinks?

In [ ]:
print("=" * 78); print("K-SWEEP RESULTS"); print("=" * 78)
summary = gen.groupby("k").agg(
    n=("qid", "count"), gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"), correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"), judge_parsed=("judge_parsed", "mean"),
).sort_index()
ans = gen[gen.refused == 0]
summary["correctness_answered"] = ans.groupby("k")["correct"].mean()
summary["faithfulness_answered"] = ans.groupby("k")["faithful"].mean()
print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv(out("k_sweep_tpu_summary.csv"))
print("""
  *_answered  same metric excluding refusals: a refusal is not "supported by the
              reference texts", so the judge scores refusals near-randomly.
""")

### Statistical test: every k vs the largest, paired

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_k, b_k, col):
    a = gen[gen.k == a_k].set_index("qid")[col]
    b = gen[gen.k == b_k].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    return d.mean(), *np.percentile(m, [2.5, 97.5])

print("\n" + "=" * 78)
print("DOES SHRINKING k IMPROVE CORRECTNESS? (paired 95% CI)")
print("=" * 78)
ks = sorted(gen.k.unique())
rows = []
for k in ks:
    if k == max(ks):
        continue
    d, lo, hi = paired(k, max(ks), "correct")
    sig = ("yes -- fewer distractors help" if lo > 0 else
           "yes -- fewer distractors HURT" if hi < 0 else "no")
    print(f"  k={k} vs k={max(ks)}   correctness diff {d:+.3f}  95%% CI [{lo:+.3f}, {hi:+.3f}]  {sig}")
    rows.append({"k": k, "vs_k": max(ks), "diff": d, "lo": lo, "hi": hi, "verdict": sig})
pd.DataFrame(rows).to_csv(out("k_sweep_tpu_comparisons.csv"), index=False)

### Verdict against the oracle hypothesis

In [ ]:
print("\n" + "=" * 78); print("VERDICT"); print("=" * 78)
best_k = summary["correctness"].idxmax()
worst_k = summary["correctness"].idxmin()
print(f"  Best correctness:  k={best_k}  ({summary.loc[best_k, 'correctness']:.3f})")
print(f"  Worst correctness: k={worst_k}  ({summary.loc[worst_k, 'correctness']:.3f})")
print(f"  Spread across k:   {summary['correctness'].max() - summary['correctness'].min():.3f}")

# Compared against the committed run, not a hardcoded number.
if "C4" in REF:
    print(f"\n  Reference oracle (1 passage, gold always present): "
          f"{REF['C4']['correctness']:.3f}")
    print(f"  This run at k=1 (1 passage, gold NOT always present): "
          f"{summary.loc[1, 'correctness']:.3f}" if 1 in summary.index else "")
    print("""
  k=1 uses one passage but does not always have the gold one, so the difference
  between it and the oracle isolates the cost of retrieval error alone, with
  context size held constant.""")
else:
    print("\n  No reference run available, so no oracle comparison is made here.")

if summary["correctness"].max() - summary["correctness"].min() < 0.02:
    print("""
  Correctness barely moves across k. On this evidence distractor count is not
  what separates the reranked condition from the oracle -- report that as the
  finding rather than looking for a k that "wins".""")

print(f"\nAll outputs written under: {OUT_DIR.resolve()}")
for f in ["k_sweep_tpu_raw.csv", "k_sweep_tpu_summary.csv", "k_sweep_tpu_comparisons.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab and aborted the notebook on the final cell.
try:
    from google.colab import files as colab_files
    for f in ["k_sweep_tpu_raw.csv", "k_sweep_tpu_summary.csv", "k_sweep_tpu_comparisons.csv"]:
        colab_files.download(out(f))
except ImportError:
    print("(Not in Colab - files are on disk at the path above.)")